In [1]:
# W8D4 - Install required packages

!pip -q install pytest mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# W8D4 - Imports

import os
import json
import math
import tempfile
from pathlib import Path

import mlflow
import pytest

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# W8D4 - Create project structure

PROJECT_DIR = Path("/content/w8d4_project")

SRC_DIR = PROJECT_DIR / "src"
TEST_DIR = PROJECT_DIR / "tests"
DOCS_DIR = PROJECT_DIR / "docs"

for folder in [SRC_DIR, TEST_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project structure created:")
print(PROJECT_DIR)
print(SRC_DIR)
print(TEST_DIR)
print(DOCS_DIR)

Project structure created:
/content/w8d4_project
/content/w8d4_project/src
/content/w8d4_project/tests
/content/w8d4_project/docs


In [4]:
# W8D4 - Main RAG-style pipeline

pipeline_code = '''
def validate_question(question):
    """Validate the user question."""
    if not isinstance(question, str):
        raise TypeError("Question must be a string.")

    if not question.strip():
        raise ValueError("Question cannot be empty.")

    return question.strip()


def retrieve_documents(question, documents, top_k=2):
    """Retrieve documents using simple keyword matching."""
    question = validate_question(question)

    if not isinstance(documents, list):
        raise TypeError("Documents must be provided as a list.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero.")

    question_words = set(question.lower().split())

    scored_documents = []

    for document in documents:
        words = set(document.lower().split())
        score = len(question_words.intersection(words))
        scored_documents.append((score, document))

    scored_documents.sort(reverse=True)

    return [doc for score, doc in scored_documents[:top_k] if score > 0]


def generate_answer(question, retrieved_documents):
    """Generate a simple answer from retrieved context."""
    question = validate_question(question)

    if not isinstance(retrieved_documents, list):
        raise TypeError("Retrieved documents must be a list.")

    if not retrieved_documents:
        return "No relevant information was found."

    context = " ".join(retrieved_documents)

    return f"Based on the retrieved information: {context}"


def run_pipeline(question, documents, top_k=2):
    """Run the complete retrieval and answer-generation pipeline."""
    retrieved = retrieve_documents(
        question,
        documents,
        top_k
    )

    answer = generate_answer(
        question,
        retrieved
    )

    return {
        "question": question,
        "retrieved_documents": retrieved,
        "answer": answer
    }
'''

pipeline_file = SRC_DIR / "rag_pipeline.py"

pipeline_file.write_text(pipeline_code)

print("Created:", pipeline_file)

Created: /content/w8d4_project/src/rag_pipeline.py


In [5]:
# W8D4 - Manual pipeline test

import sys

sys.path.append(str(SRC_DIR))

from rag_pipeline import run_pipeline

documents = [
    "Artificial intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning is a subset of artificial intelligence that learns patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "RAG combines retrieval with generation to provide answers using external context."
]

question = "What is artificial intelligence?"

result = run_pipeline(
    question,
    documents,
    top_k=2
)

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED DOCUMENTS:")
for document in result["retrieved_documents"]:
    print("-", document)

print("\nANSWER:")
print(result["answer"])

QUESTION:
What is artificial intelligence?

RETRIEVED DOCUMENTS:
- Machine learning is a subset of artificial intelligence that learns patterns from data.
- Artificial intelligence allows computers to perform tasks that normally require human intelligence.

ANSWER:
Based on the retrieved information: Machine learning is a subset of artificial intelligence that learns patterns from data. Artificial intelligence allows computers to perform tasks that normally require human intelligence.


In [6]:
# W8D4 - Automated tests

test_code = '''
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1] / "src"))

from rag_pipeline import (
    validate_question,
    retrieve_documents,
    generate_answer,
    run_pipeline
)


def test_validate_question():
    assert validate_question("What is AI?") == "What is AI?"


def test_validate_question_removes_extra_spaces():
    assert validate_question("  What is AI?  ") == "What is AI?"


def test_empty_question():
    try:
        validate_question("")
        assert False
    except ValueError:
        assert True


def test_retrieve_documents():
    documents = [
        "Artificial intelligence is a field of computer science.",
        "Machine learning learns patterns from data.",
        "Deep learning uses neural networks."
    ]

    result = retrieve_documents(
        "artificial intelligence",
        documents,
        top_k=2
    )

    assert len(result) >= 1
    assert "Artificial intelligence" in result[0]


def test_generate_answer():
    result = generate_answer(
        "What is AI?",
        ["Artificial intelligence is a field of computer science."]
    )

    assert "Artificial intelligence" in result


def test_empty_retrieval():
    result = generate_answer(
        "What is AI?",
        []
    )

    assert result == "No relevant information was found."


def test_complete_pipeline():
    documents = [
        "Artificial intelligence allows computers to perform intelligent tasks.",
        "Machine learning learns patterns from data."
    ]

    result = run_pipeline(
        "What is artificial intelligence?",
        documents,
        top_k=2
    )

    assert "question" in result
    assert "retrieved_documents" in result
    assert "answer" in result
'''

test_file = TEST_DIR / "test_pipeline.py"

test_file.write_text(test_code)

print("Created:", test_file)

Created: /content/w8d4_project/tests/test_pipeline.py


In [7]:
# W8D4 - Run pytest

!pytest -v /content/w8d4_project/tests/test_pipeline.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 7 items                                                              

w8d4_project/tests/test_pipeline.py::test_validate_question PASSED       [ 14%]
w8d4_project/tests/test_pipeline.py::test_validate_question_removes_extra_spaces PASSED [ 28%]
w8d4_project/tests/test_pipeline.py::test_empty_question PASSED          [ 42%]
w8d4_project/tests/test_pipeline.py::test_retrieve_documents PASSED      [ 57%]
w8d4_project/tests/test_pipeline.py::test_generate_answer PASSED         [ 71%]
w8d4_project/tests/test_pipeline.py::test_empty_retrieval PASSED         [ 85%]
w8d4_project/tests/test_pipeline.py::test_complete_pipeline PASSED       [100%]

============================== 7 passed in 0.03s ===============================


In [8]:
# W8D4 - MLflow tracking

mlflow.set_experiment("W8D4_Documentation_Testing")

with mlflow.start_run(run_name="W8D4_Test_Run"):

    mlflow.log_param("retrieval_method", "keyword_matching")
    mlflow.log_param("top_k", 2)

    test_result = run_pipeline(
        "What is artificial intelligence?",
        documents,
        top_k=2
    )

    mlflow.log_metric(
        "retrieved_document_count",
        len(test_result["retrieved_documents"])
    )

    mlflow.log_metric(
        "answer_length",
        len(test_result["answer"])
    )

    mlflow.log_param(
        "test_status",
        "passed"
    )

    print("MLflow run completed successfully.")

2026/09/28 09:00:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/28 09:00:47 INFO mlflow.store.db.utils: Updating database tables
2026/09/28 09:00:51 INFO mlflow.tracking.fluent: Experiment with name 'W8D4_Documentation_Testing' does not exist. Creating a new experiment.


MLflow run completed successfully.


In [9]:
# W8D4 - Test summary

test_summary = {
    "pipeline_test": "PASSED",
    "retrieval_test": "PASSED",
    "answer_generation_test": "PASSED",
    "empty_input_test": "PASSED",
    "MLflow_tracking": "PASSED"
}

for test_name, status in test_summary.items():
    print(f"{test_name}: {status}")

pipeline_test: PASSED
retrieval_test: PASSED
answer_generation_test: PASSED
empty_input_test: PASSED
MLflow_tracking: PASSED


In [10]:
# W8D4 - Create README documentation

readme = """
# W8D4 - Documentation, Testing & Code Review

## Objective

The objective of W8D4 is to document, test, and review an AI/ML pipeline using
the approved AI/ML 3M stack.

## Technologies

- Python
- Google Colab
- MLflow
- Pytest
- RAG-style retrieval pipeline

## Pipeline

The pipeline contains three major stages:

1. Question validation
2. Document retrieval
3. Answer generation

## Testing

Automated tests are implemented using pytest.

The tests verify:

- Valid question handling
- Empty question handling
- Document retrieval
- Answer generation
- Empty retrieval handling
- Complete pipeline execution

## MLflow Tracking

MLflow is used to track:

- Retrieval method
- top_k parameter
- Number of retrieved documents
- Answer length
- Test status

## Code Review

The code was reviewed for:

- Clear function names
- Modular structure
- Comments and docstrings
- Input validation
- Error handling
- Avoidance of unnecessary code
- Test coverage

## Execution Environment

The complete implementation was executed in Google Colab.

## Result

The automated tests passed successfully and the pipeline execution
was tracked using MLflow.
"""

readme_file = PROJECT_DIR / "README.md"
readme_file.write_text(readme)

print(readme)


# W8D4 - Documentation, Testing & Code Review

## Objective

The objective of W8D4 is to document, test, and review an AI/ML pipeline using
the approved AI/ML 3M stack.

## Technologies

- Python
- Google Colab
- MLflow
- Pytest
- RAG-style retrieval pipeline

## Pipeline

The pipeline contains three major stages:

1. Question validation
2. Document retrieval
3. Answer generation

## Testing

Automated tests are implemented using pytest.

The tests verify:

- Valid question handling
- Empty question handling
- Document retrieval
- Answer generation
- Empty retrieval handling
- Complete pipeline execution

## MLflow Tracking

MLflow is used to track:

- Retrieval method
- top_k parameter
- Number of retrieved documents
- Answer length
- Test status

## Code Review

The code was reviewed for:

- Clear function names
- Modular structure
- Comments and docstrings
- Input validation
- Error handling
- Avoidance of unnecessary code
- Test coverage

## Execution Environment

The complete imp

In [11]:
# W8D4 - Code review documentation

review = """
# W8D4 Code Review

## 1. Code Structure

The pipeline is divided into separate functions:

- validate_question()
- retrieve_documents()
- generate_answer()
- run_pipeline()

This makes the code easier to understand and maintain.

## 2. Readability

Functions use descriptive names and include docstrings explaining
their purpose.

## 3. Input Validation

The pipeline checks:

- Question type
- Empty questions
- Document type
- top_k value

## 4. Error Handling

Invalid inputs raise appropriate Python exceptions such as:

- TypeError
- ValueError

## 5. Testing

Automated pytest tests were added to verify the major pipeline functions.

## 6. MLflow Tracking

Important experiment parameters and metrics are logged using MLflow.

## 7. Maintainability

The implementation uses modular functions so individual components
can be changed without rewriting the entire pipeline.

## 8. Review Status

Code review completed successfully.
"""

review_file = DOCS_DIR / "code_review.md"
review_file.write_text(review)

print(review)


# W8D4 Code Review

## 1. Code Structure

The pipeline is divided into separate functions:

- validate_question()
- retrieve_documents()
- generate_answer()
- run_pipeline()

This makes the code easier to understand and maintain.

## 2. Readability

Functions use descriptive names and include docstrings explaining
their purpose.

## 3. Input Validation

The pipeline checks:

- Question type
- Empty questions
- Document type
- top_k value

## 4. Error Handling

Invalid inputs raise appropriate Python exceptions such as:

- TypeError
- ValueError

## 5. Testing

Automated pytest tests were added to verify the major pipeline functions.

## 6. MLflow Tracking

Important experiment parameters and metrics are logged using MLflow.

## 7. Maintainability

The implementation uses modular functions so individual components
can be changed without rewriting the entire pipeline.

## 8. Review Status

Code review completed successfully.



In [12]:
# W8D4 - Save test report

report = {
    "task": "W8D4 Documentation, Testing & Code Review",
    "environment": "Google Colab",
    "tests": {
        "question_validation": "PASS",
        "retrieval": "PASS",
        "answer_generation": "PASS",
        "empty_input_handling": "PASS",
        "complete_pipeline": "PASS"
    },
    "mlflow_tracking": "PASS",
    "code_review": "COMPLETED",
    "documentation": "COMPLETED"
}

report_file = DOCS_DIR / "test_report.json"

report_file.write_text(
    json.dumps(report, indent=4)
)

print(json.dumps(report, indent=4))

{
    "task": "W8D4 Documentation, Testing & Code Review",
    "environment": "Google Colab",
    "tests": {
        "question_validation": "PASS",
        "retrieval": "PASS",
        "answer_generation": "PASS",
        "empty_input_handling": "PASS",
        "complete_pipeline": "PASS"
    },
    "mlflow_tracking": "PASS",
    "code_review": "COMPLETED",
    "documentation": "COMPLETED"
}


In [13]:
# W8D4 - Verify generated project files

for path in PROJECT_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

README.md
tests/test_pipeline.py
docs/test_report.json
docs/code_review.md
src/rag_pipeline.py
src/__pycache__/rag_pipeline.cpython-313.pyc
tests/__pycache__/test_pipeline.cpython-313-pytest-8.4.2.pyc


In [14]:
# W8D4 - Final verification

required_files = [
    PROJECT_DIR / "README.md",
    SRC_DIR / "rag_pipeline.py",
    TEST_DIR / "test_pipeline.py",
    DOCS_DIR / "code_review.md",
    DOCS_DIR / "test_report.json"
]

print("=" * 60)
print("W8D4 FINAL VERIFICATION")
print("=" * 60)

all_files_exist = True

for file in required_files:
    exists = file.exists()

    print(f"{file.name}: {'PASS' if exists else 'FAIL'}")

    if not exists:
        all_files_exist = False

print("-" * 60)

if all_files_exist:
    print("Documentation: PASS")
    print("Testing: PASS")
    print("Code Review: PASS")
    print("MLflow Tracking: PASS")
    print("\nW8D4 COMPLETED SUCCESSFULLY")
else:
    print("Some files are missing.")

W8D4 FINAL VERIFICATION
README.md: PASS
rag_pipeline.py: PASS
test_pipeline.py: PASS
code_review.md: PASS
test_report.json: PASS
------------------------------------------------------------
Documentation: PASS
Testing: PASS
Code Review: PASS
MLflow Tracking: PASS

W8D4 COMPLETED SUCCESSFULLY


In [15]:
# W8D4 - Create submission ZIP

import shutil

zip_path = shutil.make_archive(
    "/content/W8D4_Documentation_Testing_CodeReview",
    "zip",
    PROJECT_DIR
)

print("ZIP created:")
print(zip_path)

ZIP created:
/content/W8D4_Documentation_Testing_CodeReview.zip
